In [8]:
import asyncio

async def hello():
    await asyncio.sleep(2)
    print("hello")

print("before")

await hello() # Do this in Notebook

print("after")

# asyncio.run(hello()) # Do this in .py file


before
hello
after


## 2. async and await

`async` tells Python that a function can perform asynchronous operations.

`await` tells Python:

"Pause this coroutine until this asynchronous operation is ready, but allow the event loop to do other work."

`await` can only be used inside an `async` function.

In [9]:
import asyncio

async def fetch_data():
    print("Fetching data...")

    await asyncio.sleep(2)

    print("Data received!")
    return "Some data"


async def main():
    data = await fetch_data()
    print(data)

await main()

Fetching data...
Data received!
Some data


In [10]:
import asyncio
import time


async def task_a():
    print("Task A started")
    await asyncio.sleep(2)
    print("Task A finished")


async def task_b():
    print("Task B started")
    await asyncio.sleep(1)
    print("Task B finished")


async def main():
    await task_a()
    await task_b()


start = time.perf_counter()

await main()

print(f"Time taken: {time.perf_counter() - start:.2f} seconds")

Task A started
Task A finished
Task B started
Task B finished
Time taken: 3.00 seconds


## 4. Concurrency with asyncio.gather()

So far, we waited for Task A to finish before starting Task B.

But if the tasks are independent, we can run them concurrently.

`asyncio.gather()` schedules multiple coroutines and waits for all of them to complete.

This is particularly useful for I/O-bound operations such as:

- API calls
- Database queries
- LLM requests
- File/network operations

In [ ]:
import asyncio
import time


async def task_a():
    print("Task A started")
    await asyncio.sleep(2)
    print("Task A finished")

    return "Result A"


async def task_b():
    print("Task B started")
    await asyncio.sleep(1)
    print("Task B finished")

    return "Result B"


async def main():
    results = await asyncio.gather(
        task_a(),
        task_b(),
    )

    print(results)


start = time.perf_counter()

await main()

# asyncio.run(main())

print(f"Time taken: {time.perf_counter() - start:.2f} seconds")

Task A started
Task B started
Task B finished
Task A finished
['Result A', 'Result B']
Time taken: 2.00 seconds


## 5. The Event Loop

The event loop is the engine behind asyncio.

It keeps track of asynchronous tasks and resumes them when they are ready to make progress.

When one task is waiting for I/O, the event loop can work on another task. You normally don't need to manually manage the event loop.

For standalone Python programs, `asyncio.run()` is the common entry point. But a Jupyter/VS Code notebook kernel is already running inside its own asyncio event loop (that's how the kernel manages cells under the hood). Notebooks support top-level await for exactly this reason — there's already a loop running, so you hook into it instead of starting a new one.

`asyncio.run()` is for plain .py scripts run from the terminal, where no loop exists yet and something has to create one — that's the if `__name__ == "__main__": asyncio.run(main())` pattern.

## 7. Async Generators and async for

An async generator can produce values asynchronously over time.

We use:

    async def
    yield

to create an async generator.

We consume it using:

    async for

This is particularly useful for streaming responses from LLMs or other APIs.

In [15]:
import asyncio

async def generate_numbers():

    for number in range(5):
        await asyncio.sleep(2)
        yield number


async def main():

    async for number in generate_numbers():
        print(number)


await main()

0
1
2
3
4


In [16]:
import asyncio
import httpx


async def fetch_post():

    async with httpx.AsyncClient() as client:

        response = await client.get(
            "https://jsonplaceholder.typicode.com/posts/1"
        )

        return response.json()


async def main():
    post = await fetch_post()
    print(post)


await main()

{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


In [ ]:
import asyncio
import httpx


async def fetch_user(client):
    response = await client.get(
        "https://jsonplaceholder.typicode.com/users/1"
    )

    return response.json()


async def fetch_post(client):
    response = await client.get(
        "https://jsonplaceholder.typicode.com/posts/1"
    )

    return response.json()


async def fetch_todo(client):
    response = await client.get(
        "https://jsonplaceholder.typicode.com/todos/1"
    )

    return response.json()


async def main():

    async with httpx.AsyncClient() as client:

        user, post, todo = asyncio.gather(
            fetch_user(client),
            fetch_post(client),
            fetch_todo(client),
        )

    print("User:")
    print(user["name"])

    print("\nPost:")
    print(post["title"])

    print("\nTodo:")
    print(todo["title"])


await main()

after


Here is the consolidated Markdown content cleaned up into a single, cohesive document:

# 📝 Homework

## Homework 1: Async Functions and `await`

Create an async function called `process_task()` that simulates a task taking 2 seconds to complete.

Inside the function:

1. Print `"Task started"`
2. Wait asynchronously for 2 seconds using `asyncio.sleep()`
3. Print `"Task completed"`
4. Return `"Success"`

Create an `async main()` function that calls `process_task()` and prints the returned result.

### Expected Output

```text
Task started
Task completed
Success

```

### Goal

Understand:

* `async def`
* `await`
* `asyncio.sleep()`
* How an async function is executed using `asyncio.run()`

---

## Homework 2: Concurrency with `asyncio.gather()`

You have three independent tasks:

* Downloading a document → 3 seconds
* Generating embeddings → 2 seconds
* Fetching metadata → 1 second

Create three async functions that simulate these operations using `asyncio.sleep()`.

Then use `asyncio.gather()` to execute all three tasks concurrently.

Measure the total execution time using `time.perf_counter()`.

### Requirements

Print the result of each task and the total execution time.

### Expected Observation

If the tasks were executed sequentially, the total time would be approximately:

```text
3 + 2 + 1 = 6 seconds

```

With `asyncio.gather()`, the total time should be approximately:

```text
3 seconds

```

### Goal

Understand how asynchronous concurrency can reduce the total waiting time for independent I/O-bound operations.

---

## Homework 3: Concurrent API Requests

Use `httpx.AsyncClient` and the JSONPlaceholder API.

Fetch the following three resources concurrently:

```text
https://jsonplaceholder.typicode.com/users/1
https://jsonplaceholder.typicode.com/posts/1
https://jsonplaceholder.typicode.com/todos/1

```

### Requirements

1. Create an async function called `fetch_data(url)`.
2. Use `httpx.AsyncClient` to make the HTTP request.
3. Return the JSON response.
4. Use `asyncio.gather()` to call all three URLs concurrently.
5. Print the important information from each response.

For example:

```text
User: Leanne Graham
Post ID: 1
Todo: delectus aut autem

```

### Goal

Combine the concepts you have learned:

* `async def`
* `await`
* `httpx.AsyncClient`
* `asyncio.gather()`
* Concurrent I/O

---

## Homework 4: Build an Async API Client

Create a small `PostService` class that communicates with the JSONPlaceholder API.

The class should contain an async method:

```python
async def get_post(self, post_id):
    ...

```

The method should:

1. Accept a `post_id`.
2. Make an asynchronous HTTP request using `httpx.AsyncClient`.
3. Fetch the corresponding post from:

```text
https://jsonplaceholder.typicode.com/posts/{post_id}

```

4. Return the JSON response.

Then fetch posts with IDs `1`, `2`, and `3` concurrently using `asyncio.gather()`.

### Expected Output

Print something similar to:

```text
Post 1: ...
Post 2: ...
Post 3: ...

```

### Bonus

Add basic error handling:

* If the API returns a non-success status code, raise an exception.
* Handle the exception in `main()` and print a meaningful error message.

### Goal

Practice structuring asynchronous HTTP code into a reusable service, similar to how you will later interact with LLM APIs.

---

## Homework 5: Build a Concurrent LLM-Style Data Fetcher

Imagine that your AI application needs information from three independent sources before calling an LLM:

* User profile
* User's posts
* User's todos

Use the JSONPlaceholder API to simulate these three data sources.

Create three async functions:

```python
async def get_user():
    ...

async def get_posts():
    ...

async def get_todos():
    ...

```

Each function should make an HTTP request using `httpx.AsyncClient`.

Then create an `async main()` function that:

1. Starts all three requests concurrently using `asyncio.gather()`.
2. Waits for all three responses.
3. Combines the results into a single Python dictionary.

The final structure should look similar to:

```python
{
    "user": {...},
    "posts": [...],
    "todos": [...]
}

```

Print the final dictionary.

### Bonus

Measure the total execution time and compare it with a sequential implementation.

### Goal

Think about how async concurrency is used in real AI systems, where an application may need to fetch multiple pieces of information before sending a request to an LLM.